In [ ]:
import json
from datasets import load_dataset

meu_dataset = load_dataset('json', data_files={
    'train': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/train (1).json',
    'validation': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/val.json',
    'test': '/home/cecilia/Documentos/PIBIC/Fase2/DADOS-LLMS/test (1).json'
})


In [ ]:
import pandas as pd
df_train = pd.DataFrame(meu_dataset['train'])

for i, row in df_train.head(20).iterrows():
    print(f"{i}: {row['sentence'][:80]}")
    print(f"   {row['rotulos']}")
    print()

In [ ]:
import os
if os.path.exists(caminho_salvar):
    os.remove(caminho_salvar)
    print("Arquivo deletado!")
else:
    print("Arquivo não existe")

In [ ]:
import re

def extrair_linhas_validas(texto_da_api: str) -> list:
    
    padrao = r'([^|\n"\s]+(?: [^|\n"\s]+)*)\|([A-Za-z0-9_]+#[A-Za-z0-9_]+)'

    matches = re.findall(padrao, texto_da_api)

    linhas_validas = []
    for match in matches:
        aspecto = match[0].strip()
        categoria = match[1].strip()
        linhas_validas.append(f"{aspecto}|{categoria}")

    if linhas_validas:
        return list(dict.fromkeys(linhas_validas))

    if "null" in texto_da_api.lower() or "none" in texto_da_api.lower():
        return ["None"]

    return []

In [ ]:
from openai import OpenAI
import pandas as pd
from typing import List, Dict, Any
import time
from tqdm.notebook import tqdm
import os

class OpenRouterBatchInference:
    def __init__(self, api_key: str, models: List[str], system_prompt: str): 
      
        self.client = OpenAI(
            base_url="https://openrouter.ai/api/v1",
            api_key=api_key
        )
        self.models = models
        self.system_prompt = system_prompt


    def _create_messages(self, user_prompt: str) -> List[Dict[str, str]]:
        
        return [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": f"Input: {user_prompt}\nOutput:"}
        ]



    def _query_model(self, model: str, user_prompt: str) -> str:
       
        completion = self.client.chat.completions.create(
            model=model,
            messages=self._create_messages(user_prompt),
            timeout=120,
            max_tokens=1000,
            temperature=0.0
        )

        resposta_bruta = completion.choices[0].message.content if completion.choices[0].message.content else ""

        linhas_filtradas = extrair_linhas_validas(resposta_bruta)
        resultado_final = "\n".join(linhas_filtradas)

        print("Resultado limpo extraído:", resultado_final)

        if resultado_final.strip():
            return resultado_final

        if hasattr(completion.choices[0].message, 'reasoning') and completion.choices[0].message.reasoning:
            return completion.choices[0].message.reasoning

        return "ERRO: output vazio ou formato inválido"

    def generate_outputs(self, dataset: pd.DataFrame, save_path: str) -> Dict[str, List[Dict[str, Any]]]: 
        all_outputs = {model: [] for model in self.models}

        if os.path.exists(save_path):
          with open(save_path, 'r', encoding='utf-8') as f:
            all_outputs = json.load(f)
            print("Progresso anterior carregado com sucesso!")

        print("Montando os índices já processados na memória... Aguarde.")
        indices_processados = {
            model: set(int(item["index"]) for item in all_outputs[model] if "index" in item)
            for model in self.models
        }

        loop_progresso = tqdm(dataset.index, desc="Processando dataset")

        
        for index in loop_progresso:
          data_point = dataset.loc[index]
          user_prompt = f"Input: {data_point['sentence']}"

          if all(int(index) in indices_processados[model] for model in self.models):
                continue

          for model in self.models:

            if int(index) in indices_processados[model]:
                    continue

            
            try:
              loop_progresso.set_postfix(modelo=model)
              output = self._query_model(model, user_prompt)

              all_outputs[model].append({
                  "index": int(index),
                  "input": data_point.to_dict(),
                  "output": output
              })

            except Exception as e:
              print(f"\n[Erro] no modelo {model} no índice {index}: {e}")
              all_outputs[model].append({
                  "index": int(index),
                  "input": data_point.to_dict(),
                  "output": "null|Error#API"
              })


            time.sleep(3)

          with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(all_outputs, f, ensure_ascii=False, indent=4)

        return all_outputs


In [15]:
df_teste = pd.DataFrame(meu_dataset['test'])
df_val = pd.DataFrame(meu_dataset['validation'])

In [ ]:
caminho_salvar = '/home/cecilia/Documentos/PIBIC/Fase2/Resultados_Modelos/resultado_nvidia_final.json'



In [ ]:
api_key = ''

models = [
  'nvidia/nemotron-3-ultra-550b-a55b:free'
]

In [ ]:
# System prompt
system_prompt = ("""
You are an expert NLP model specialized in Aspect-Based Sentiment Analysis (ABSA).
Your task is to extract aspect terms and their corresponding entity-attribute categories from the user's text based STRICTLY on the taxonomy provided below.

#### ALLOWED TAXONOMY (ENTITY#ATTRIBUTE) ####
You must ONLY use the categories listed below. Do not invent or use any other combination.

- HOTEL & ACCOMMODATION:
  Hotel#General, Hotel#Comfort, Hotel#Cleanliness, Hotel#Design_features,
  Rooms#General, Rooms#Comfort, Rooms#Cleanliness, Rooms#Design_features,
  Room_Amenities#General, Room_Amenities#Design_features, Location#General

- RESTAURANT / FOOD & DRINKS:
  Restaurant#General, Restaurant#Miscellaneous, Restaurant#Prices,
  Food#Quality, Food#Style_Options, Drinks#Quality, Ambience#General

- COMPUTERS & HARDWARE:
  LAPTOP#GENERAL, LAPTOP#QUALITY, LAPTOP#PRICE, LAPTOP#DESIGN_FEATURES, LAPTOP#OPERATION_PERFORMANCE,
  DISPLAY#GENERAL, DISPLAY#QUALITY, DISPLAY#DESIGN_FEATURES, DISPLAY#OPERATION_PERFORMANCE,
  KEYBOARD#GENERAL, KEYBOARD#DESIGN_FEATURES, KEYBOARD#OPERATION_PERFORMANCE,
  BATTERY#GENERAL, BATTERY#OPERATION_PERFORMANCE,
  MULTIMEDIA_DEVICES#OPERATION_PERFORMANCE

- BOOKS & CONTENT:
  Book#General, Book#Quality, Book#Author, Content#Plot, Content#Characters

- CLOTHING & FOOTWEAR:
  Shoes#General, Shoes#Quality, Shoes#Size, Shoes#Looking,
  Clothing#General, Clothing#Quality, Clothing#Size,
  Top#General, Top#Quality,
  Bottom#General, Bottom#Quality, Bottom#Size, Bottom#Looking

- SERVICES:
  Service#General

#### CONSTRAINTS & FORMATTING RULES ####
1. For each aspect found, respond strictly in the format: aspect_term|ENTITY#ATTRIBUTE
2. Respond ONLY with the extracted pairs. No introduction, no explanation, no markdown, no extra text before or after.
3. If multiple aspects are present, separate them with a comma and a space (e.g., term1|ENTITY#ATTRIBUTE, term2|ENTITY#ATTRIBUTE).
4. Match the category casing EXACTLY as shown in the taxonomy (e.g., 'LAPTOP#GENERAL' in uppercase, 'Hotel#General' in mixed case).
5. Use 'NULL' ONLY when the text has absolutely no word or pronoun referring to the aspect.
6. If no aspects from the allowed taxonomy are found, respond with: None



#### REFERENCE EXAMPLES ####

Example 1:
Input: The staff is friendly.
Output: staff|Service#General

Example 2:
Input: Awesome place and an awesome host!
Output: place|Hotel#General, host|Service#General

Example 3:
Input: In my opinion she has become a page-turner author and I just couldn't put this book down.
Output: author|Book#Author, book|Book#General

Example 4:
Input: Delicious.
Output: null|Food#Quality

Example 5:
Input: this product seemed perfect for me.
Output: product|LAPTOP#GENERAL

Example 6:
Input: They have always been just a tad tight.
Output: They|Shoes#Size

Example 7:
Input: Going back soon ( who doesnt like $ 1 draft beer ).
Output: NULL|Restaurant#General
"""
)


inference = OpenRouterBatchInference(
    api_key=api_key,
    models=models,
    system_prompt=system_prompt
)



In [7]:
import regex as re
def calcular_metricas(gabarito_set_ou_str, ia_output):

    if ia_output is None or isinstance(ia_output, dict) or str(ia_output).startswith("ERRO"):
      return {"Precisao": 0.0, "Recall": 0.0, "F1": 0.0, "Acuracia": 0.0}

    gabarito_str = str(gabarito_set_ou_str).lower()
    ia_str = str(ia_output).lower()

    ia_str = re.sub(r'\bimplicit\b', 'null', ia_str)

    def extrair_pares(texto):
        for char in ["{", "}", "[", "]", "'", '"']:
            texto = texto.replace(char, "")

        pares = set()
        itens = texto.split(',')
        for item in itens:
            item = item.strip()
            if '|' in item:
                item = re.sub(r'\s+', '', item)

                pares.add(item)
        return pares

    set_gabarito = extrair_pares(gabarito_str)
    set_ia = extrair_pares(ia_str)

    if not set_gabarito and not set_ia:
        return {"Precisao": 1.0, "Recall": 1.0, "F1": 1.0, "Acuracia": 1.0}

    if not set_gabarito or not set_ia:
        return {"Precisao": 0.0, "Recall": 0.0, "F1": 0.0, "Acuracia": 0.0}

    acertos = set_gabarito.intersection(set_ia)
    n_acertos = len(acertos)

    recall = n_acertos / len(set_gabarito)
    precisao = n_acertos / len(set_ia) if len(set_ia) > 0 else 0
    f1 = (2 * precisao * recall) / (precisao + recall) if (precisao + recall) > 0 else 0


    uniao = set_gabarito.union(set_ia)
    acuracia = n_acertos / len(uniao) if len(uniao) > 0 else 0

    return {"Precisao": precisao, "Recall": recall, "F1": f1, "Acuracia": acuracia}

In [ ]:
modelo = models[0]


In [ ]:
import json

caminho = '/home/cecilia/Documentos/PIBIC/Fase2/Resultados_Modelos/resultado_nvidia_final.json'

with open(caminho) as f:
    outputs = json.load(f)

modelo = list(outputs.keys())[0]

validos_antes = len(outputs[modelo])

outputs[modelo] = [
    item for item in outputs[modelo]
    if not (item.get('index') >= 1101 and item.get('output') == 'null|Error#API')
    and item.get('index') != 1328
]

validos_depois = len(outputs[modelo])
print(f'Removidos: {validos_antes - validos_depois}')
print(f'Válidos restantes: {validos_depois}')

with open(caminho, 'w', encoding='utf-8') as f:
    json.dump(outputs, f, ensure_ascii=False, indent=4)

print('Arquivo limpo e salvo!')

In [ ]:
modelo = "nvidia/nemotron-3-ultra-550b-a55b:free"

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd


outputs = inference.generate_outputs(df_teste, caminho_salvar)

for i, item in enumerate(outputs[modelo]):
    print(f"--- Item {i} ---")
    print("Output:", repr(item['output']))
    print()


item = outputs[modelo][0]
print("Output bruto:", repr(item['output']))
resultados_pibic_final = []



modelo = models[0]

for i, item in enumerate(outputs[modelo]):
    frase_original = item['input']['sentence']
    gabarito_oficial = item['input']['rotulos']
    predicao = item['output']

    metricas = calcular_metricas(gabarito_oficial, predicao)
    resultados_pibic_final.append({
        "Frase": frase_original,
        "IA": predicao,
        "Gabarito": gabarito_oficial,
        "Precisao": metricas["Precisao"],
        "Recall": metricas["Recall"],
        "F1": metricas["F1"],
        "Acuracia": metricas["Acuracia"]
    })


df_final = pd.DataFrame(resultados_pibic_final)
print("\n--- MÉDIAS FINAIS DO EXPERIMENTO ---")
print(df_final[['Precisao', 'Recall', 'F1', 'Acuracia']].mean())

In [ ]:
import json
import time
from openai import RateLimitError

caminho_do_backup_original = caminho_salvar.replace(".json", "_corrigido.json")
caminho_salvar_novo = caminho_do_backup_original

try:
    with open(caminho_do_backup_original, 'r', encoding='utf-8') as f:
        historico = json.load(f)
except FileNotFoundError:
    caminho_do_backup_original = caminho_salvar
    with open(caminho_do_backup_original, 'r', encoding='utf-8') as f:
        historico = json.load(f)

modelo_atual = "nvidia/nemotron-3-ultra-550b-a55b:free"
predicoes_salvas = historico.get(modelo_atual, [])

print(f"Iniciando o pente-fino blindado. Salvando em: {caminho_salvar_novo}")

falhas_formato_e_vazio = ["None", "ERRO: output vazio", "ERRO: output vazio ou formato inválido", None]

for idx, item in enumerate(predicoes_salvas):
    predicao = item.get('output', None)
    if isinstance(predicao, str):
        predicao = predicao.strip()

    precisa_reprocessar = False
    motivo = ""

    if 922 <= idx <= 1006:
        if predicao not in falhas_formato_e_vazio and "ERRO" not in str(predicao):
            continue
        precisa_reprocessar = True
        motivo = "Queda do Servidor / Limite Diário Antigo"

    elif predicao in falhas_formato_e_vazio:
        precisa_reprocessar = True
        motivo = "Alucinação de Formato (None)"

    if not precisa_reprocessar:
        continue

    print(f"\n⚡ Corrigindo índice {idx} -> Motivo: {motivo}")
    frase_original = item.get('input', {}).get('sentence', '')
    prompt_formatado = f"Input: {frase_original}"

    try:
        nova_resposta = inference._query_model(model=modelo_atual, user_prompt=prompt_formatado)
        print(f"   ↳ Resposta bruta da IA: {nova_resposta}")

        if isinstance(nova_resposta, str) and "->" in nova_resposta:
            print("   Ajustando formato de setinha (->) para barra (|)...")
            nova_resposta = nova_resposta.replace("->", "|")
            nova_resposta = nova_resposta.replace("  |  ", " | ").replace(" |  ", " | ").replace("  | ", " | ")
            print(f"   ↳ Resposta ajustada: {nova_resposta}")

        item['output'] = nova_resposta

    except RateLimitError as e:
        print(f"\nLimite diário da chave atingido no índice {idx}!")
        break

    except Exception as e:
        print(f"    Falha na chamada da API para o índice {idx}. Erro capturado: {e}")
        print("   Pulando esta linha para não travar a execução...")
        item['output'] = "ERRO: Instabilidade na API"
        continue

    with open(caminho_salvar_novo, 'w', encoding='utf-8') as f:
        json.dump(historico, f, ensure_ascii=False, indent=4)

print("\n Execução finalizada! O arquivo está atualizado com segurança.")

In [ ]:
import json

caminho_do_backup = caminho_salvar.replace(".json", "_corrigido.json")

with open(caminho_do_backup, 'r', encoding='utf-8') as f:
    historico = json.load(f)

modelo_atual = "nvidia/nemotron-3-ultra-550b-a55b:free"
predicoes_salvas = historico.get(modelo_atual, [])

valores_erro_api = ["ERRO: 429 - Limite Diário Excedido", "ERRO: Instabilidade na API"]

print("Buscando os 2 buracos gerados por erro de API...\n")
indices_com_erro = []

for idx, item in enumerate(predicoes_salvas):
    predicao = item.get('output', None)

    if predicao in valores_erro_api:
        indices_com_erro.append(idx)
        frase = item.get('input', {}).get('sentence', '')
        print(f"Encontrado! Índice: {idx}")
        print(f"   Frase: '{frase}'\n")

print(f"Lista de índices para corrigir: {indices_com_erro}")

In [ ]:
for idx in indices_com_erro:
    item = predicoes_salvas[idx]
    frase_original = item.get('input', {}).get('sentence', '')
    prompt_formatado = f"Input: {frase_original}"

    print(f"⚡ Tentando recuperar índice {idx}...")
    try:
        nova_resposta = inference._query_model(model=modelo_atual, user_prompt=prompt_formatado)
        print(f"   ↳ Resposta da IA: {nova_resposta}")

        if isinstance(nova_resposta, str) and "->" in nova_resposta:
            nova_resposta = nova_resposta.replace("->", "|")
            nova_resposta = nova_resposta.replace("  |  ", " | ").replace(" |  ", " | ").replace("  | ", " | ")

        item['output'] = nova_resposta

    except Exception as e:
        print(f"    Erro ao tentar rodar o índice {idx} novamente: {e}")

with open(caminho_do_backup, 'w', encoding='utf-8') as f:
    json.dump(historico, f, ensure_ascii=False, indent=4)

print("\n Pronto! As duas frases foram processadas e salvas com sucesso.")

In [ ]:
import json
import pandas as pd

caminho_do_backup = caminho_salvar.replace(".json", "_corrigido.json")

try:
    with open(caminho_do_backup, 'r', encoding='utf-8') as f:
        historico = json.load(f)

    modelo_atual = "nvidia/nemotron-3-ultra-550b-a55b:free"
    predicoes_salvas = historico.get(modelo_atual, [])

    resultados_parciais = []
    total_no_arquivo = len(predicoes_salvas)

    erros_reais_api_servidor = 0
    predicoes_avaliadas = 0

    for idx, item in enumerate(predicoes_salvas):
        predicao = item.get('output', None)

        if isinstance(predicao, str):
            predicao = predicao.strip()

        valores_erro_modelo = ["None", "ERRO: output vazio", "ERRO: output vazio ou formato inválido", None]

        valores_erro_api = ["ERRO: 429 - Limite Diário Excedido", "ERRO: Instabilidade na API"]

        if predicao in valores_erro_api:
            erros_reais_api_servidor += 1
            continue

        if predicao in valores_erro_modelo:
            predicao = "None"

        input_dados = item.get('input', {})
        frase_original = input_dados.get('sentence', '')
        gabarito_oficial = input_dados.get('rotulos', '')

        func_metrica = globals().get('calcular_metrics') or globals().get('calcular_metrica') or globals().get('calcular_metricas')
        metricas = func_metrica(gabarito_oficial, predicao)

        resultados_parciais.append({
            "Frase": frase_original,
            "IA": predicao,
            "Gabarito": gabarito_oficial,
            "Precisao": metricas["Precisao"],
            "Recall": metricas["Recall"],
            "F1": metricas["F1"],
            "Acuracia": metricas["Acuracia"]
        })

    df_teste_parcial = pd.DataFrame(resultados_parciais)

    print(f"Total de registros no backup: {total_no_arquivo}")
    print(f"Descartados da métrica (Buracos de API/Servidor não respondidos): {erros_reais_api_servidor}")
    print(f"Amostras efetivamente avaliadas (Dados Reais + Alucinações Tratadas): {len(df_teste_parcial)}")

    print(f"\n--- MÉDIAS GLOBAIS ATUALIZADAS DO MODELO (PIBIC) ---")
    if len(df_teste_parcial) > 0:
        print(df_teste_parcial[['Precisao', 'Recall', 'F1', 'Acuracia']].mean())
    else:
        print("Nenhuma amostra válida para calcular.")

except FileNotFoundError:
    print(f"Erro: O arquivo '{caminho_do_backup}' não foi encontrado. Rode o pente-fino primeiro!")
except Exception as e:
    print(f"Ocorreu um erro ao ler o backup: {e}")